In [1]:
!pip install pillow ipython ipywidgets


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\el\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from PIL import Image, ImageDraw
from IPython.display import display, clear_output
import ipywidgets as widgets


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
import ipywidgets as widgets
from io import BytesIO

ROWS, COLS = 3, 3
CELL_SIZE = 120
WIDTH, HEIGHT = COLS*CELL_SIZE, ROWS*CELL_SIZE

BG_COLOR = (255, 255, 255)
LINE_COLOR = (180, 180, 180)
PLAYER_O_COLOR = (255, 105, 180)
AI_X_COLOR = (30, 144, 255)

LINE_WIDTH = 6
CIRCLE_WIDTH = 6
SPACE = 25

board = [['' for _ in range(COLS)] for _ in range(ROWS)]
game_over = False

def check_win(player):
    for i in range(3):
        if all(board[i][j] == player for j in range(3)): return True
        if all(board[j][i] == player for j in range(3)): return True
    if all(board[i][i] == player for i in range(3)): return True
    if all(board[i][2-i] == player for i in range(3)): return True
    return False

def check_draw():
    return all(board[r][c] != '' for r in range(ROWS) for c in range(COLS))

def minimax(board_state, is_max):
    if check_win('X'): return 1
    if check_win('O'): return -1
    if check_draw(): return 0

    if is_max:
        best = -999
        for r in range(3):
            for c in range(3):
                if board_state[r][c] == '':
                    board_state[r][c] = 'X'
                    score = minimax(board_state, False)
                    board_state[r][c] = ''
                    best = max(best, score)
        return best
    else:
        best = 999
        for r in range(3):
            for c in range(3):
                if board_state[r][c] == '':
                    board_state[r][c] = 'O'
                    score = minimax(board_state, True)
                    board_state[r][c] = ''
                    best = min(best, score)
        return best

def best_move():
    best_score = -999
    move = None
    for r in range(3):
        for c in range(3):
            if board[r][c] == '':
                board[r][c] = 'X'
                score = minimax(board, False)
                board[r][c] = ''
                if score > best_score:
                    best_score = score
                    move = (r, c)
    return move

def draw_board():
    img = Image.new("RGB", (WIDTH, HEIGHT), BG_COLOR)
    draw = ImageDraw.Draw(img)

    for i in range(1, 3):
        draw.line((0, i*CELL_SIZE, WIDTH, i*CELL_SIZE), fill=LINE_COLOR, width=LINE_WIDTH)
        draw.line((i*CELL_SIZE, 0, i*CELL_SIZE, HEIGHT), fill=LINE_COLOR, width=LINE_WIDTH)

    for r in range(3):
        for c in range(3):
            x0 = c*CELL_SIZE + SPACE
            y0 = r*CELL_SIZE + SPACE
            x1 = (c+1)*CELL_SIZE - SPACE
            y1 = (r+1)*CELL_SIZE - SPACE

            if board[r][c] == 'O':
                draw.ellipse([x0, y0, x1, y1], outline=PLAYER_O_COLOR, width=CIRCLE_WIDTH)
            elif board[r][c] == 'X':
                draw.line([x0, y0, x1, y1], fill=AI_X_COLOR, width=LINE_WIDTH)
                draw.line([x0, y1, x1, y0], fill=AI_X_COLOR, width=LINE_WIDTH)
    return img

# ---------- التعديل هنا ----------
board_image = widgets.Image(format='png')

def update_display():
    img = draw_board()
    buffer = BytesIO()
    img.save(buffer, format='PNG')
    board_image.value = buffer.getvalue()
# --------------------------------

buttons = [[widgets.Button(
    description="",
    layout=widgets.Layout(width='100px', height='100px'),
    style={'button_color': '#fce4ec'}
) for c in range(3)] for r in range(3)]

status_area = widgets.Output()

def on_click(b, row, col):
    global board, game_over
    if board[row][col] == "" and not game_over:
        board[row][col] = "O"
        update_display()

        if check_win('O'):
            game_over = True
            with status_area:
                status_area.clear_output()
                print("Player O wins!")
            return

        if check_draw():
            game_over = True
            with status_area:
                status_area.clear_output()
                print("Draw")
            return

        move = best_move()
        if move:
            board[move[0]][move[1]] = "X"
        update_display()

        if check_win('X'):
            game_over = True
            with status_area:
                status_area.clear_output()
                print("AI X wins!")

for r in range(3):
    for c in range(3):
        buttons[r][c].on_click(lambda b, row=r, col=c: on_click(b, row, col))

reset_button = widgets.Button(
    description="Reset",
    layout=widgets.Layout(width='320px', height='50px'),
    style={'button_color': '#ffe082'}
)

def reset_game(b):
    global board, game_over
    board = [['' for _ in range(COLS)] for _ in range(ROWS)]
    game_over = False
    with status_area:
        status_area.clear_output()
    update_display()

reset_button.on_click(reset_game)

grid = widgets.GridBox(
    [btn for row in buttons for btn in row],
    layout=widgets.Layout(grid_template_columns="repeat(3, 100px)")
)

display(grid, board_image, status_area, reset_button)
update_display()


GridBox(children=(Button(layout=Layout(height='100px', width='100px'), style=ButtonStyle(button_color='#fce4ec…

Image(value=b'')

Output()

Button(description='Reset', layout=Layout(height='50px', width='320px'), style=ButtonStyle(button_color='#ffe0…